In [23]:
# Quick restart: load game_week_scores, game_weeks, and profiles, then merge and preview
import pandas as pd
from pathlib import Path
from IPython.display import display

base = Path('../CSV/06.04.2026')
paths = {
    'game_week_scores': base / 'game_week_scores_rows.csv',
    'game_weeks': base / 'game_weeks_rows.csv',
    'profiles': Path('../CSV/19.01.2026/profiles.csv'),
}

def load_csv(p):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f'File not found: {p}')
    df = pd.read_csv(p, dtype=str)
    df = df.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)
    for c in df.columns:
        cl = c.lower()
        if any(k in cl for k in ('date','time','created_at','updated_at','predictions_close')):
            df[c] = pd.to_datetime(df[c], errors='coerce')
    return df

gw_scores = load_csv(paths['game_week_scores'])
gws = load_csv(paths['game_weeks'])
profiles = load_csv(paths['profiles'])

# Merge game_weeks info into game_week_scores on id -> game_week_id
if 'game_week_id' in gw_scores.columns:
    merged_gw = gw_scores.merge(gws[['id','season_id','week_number']], left_on='game_week_id', right_on='id', how='left')
elif 'game_week' in gw_scores.columns:
    merged_gw = gw_scores.merge(gws[['id','season_id','week_number']], left_on='game_week', right_on='id', how='left')
else:
    raise KeyError('game_week_scores missing join key (game_week_id/game_week)')

# Merge profiles into the merged game-week scores on player_id -> id
if 'player_id' in merged_gw.columns:
    merged_all = merged_gw.merge(profiles[['id','username']], left_on='player_id', right_on='id', how='left')
    # drop the duplicate profile id column
    if 'id' in merged_all.columns:
        merged_all = merged_all.drop(columns=['id'])
else:
    raise KeyError('game_week_scores missing player_id column')

print('merged_gw shape:', merged_gw.shape)
print('merged_all (with profiles) shape:', merged_all.shape)
print('game_weeks shape:', gws.shape)
print('profiles shape:', profiles.shape)

display(merged_all.head())


merged_gw shape: (1021, 9)
merged_all (with profiles) shape: (1021, 10)
game_weeks shape: (30, 8)
profiles shape: (48, 4)


,id_x,game_week_id,player_id,correct_scores,points,created_at,id_y,season_id,week_number,username
0,00451e60-0c2b-4a2f-83e3-3968a25d92ee,3d3498a4-2cb9-46cc-a1c2-2cb731ca4d02,25472e3e-4928-440c-84a8-ed85a6e4cf34,0,1,2026-02-08 20:06:14.425546+00:00,3d3498a4-2cb9-46cc-a1c2-2cb731ca4d02,27404307-72b4-4518-a32c-ce15384915d9,23,Matt Lavery
1,006f7c5f-b47f-432f-b3aa-80e116c635b9,5e429b8e-0feb-41e8-a512-cd6ae112b657,298651b6-b5d4-4875-a636-799edae79a84,1,4,2026-03-23 01:11:21.652233+00:00,5e429b8e-0feb-41e8-a512-cd6ae112b657,27404307-72b4-4518-a32c-ce15384915d9,29,KAV
2,007143d3-0751-4a96-8082-0b266a947874,d6d48c4a-c6a6-4149-a03a-6a45d592346a,857f0ddb-bbf7-490f-89e7-c393913d1937,1,6,2026-04-06 01:37:36.003084+00:00,d6d48c4a-c6a6-4149-a03a-6a45d592346a,27404307-72b4-4518-a32c-ce15384915d9,30,Jim Shirley
3,01a27a78-5a8e-4a07-a5cb-c4e025297918,39344544-7804-4c1d-a17b-a1aff7a7f869,4f4c26c5-21c6-4a45-b9ab-141ea2936063,0,7,2025-11-10 03:54:43.745207+00:00,39344544-7804-4c1d-a17b-a1aff7a7f869,27404307-72b4-4518-a32c-ce15384915d9,11,RogerStanton
4,01ce34c2-37ab-4c9b-ab23-ef007a18460c,3715512f-bc25-48aa-a319-f9b2ffc2fb15,c8b7ea0d-f644-44ff-b89b-a0e352ff3cec,0,5,2025-10-27 01:59:34.334362+00:00,3715512f-bc25-48aa-a319-f9b2ffc2fb15,27404307-72b4-4518-a32c-ce15384915d9,9,The General


In [24]:
# Player totals: sum points and correct_scores per player from merged_all
import pandas as pd
from IPython.display import display

if 'merged_all' not in globals():
    raise NameError("merged_all not found — run cell 1 first to load and merge game_week_scores, game_weeks, and profiles")

df = merged_all.copy()
for c in ('points','correct_scores'):
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
    else:
        df[c] = 0

totals = df.groupby('player_id', dropna=False).agg({'points':'sum','correct_scores':'sum'}).reset_index()

# Attach username when available
if 'username' in df.columns:
    totals = totals.merge(df[['player_id','username']].drop_duplicates('player_id'), on='player_id', how='left')

totals = totals.rename(columns={'points':'total_points','correct_scores':'total_correct'})

# Sort by points descending
totals = totals.sort_values('total_points', ascending=False).reset_index(drop=True)

# Make tidy integer display
for col in ('total_points','total_correct'):
    totals[col] = totals[col].round(0).astype(int)

display(totals)


,player_id,total_points,total_correct,username
0,a98cac4a-9401-4a97-b8e6-93a7caee47e9,237,38,Gerard
1,3d18f1c8-de23-4c6d-b843-8241da5d994e,226,39,🍺The Barman
2,c8b7ea0d-f644-44ff-b89b-a0e352ff3cec,222,31,The General
3,b782ca87-97a5-4e94-96b3-fda9d7a93d5d,213,30,Phil Huffer
4,49f48e02-6af0-4107-8f27-29f67efc4c3a,209,26,Bob sullivan
5,6e5a68ef-c26c-4776-8f8c-7f3c22bc3f06,209,27,Nick Arnold
6,6bf245f8-9743-4b7b-ab78-dc0cdd591c6c,208,29,CFC Stew
7,95f43b8d-de0d-4dd2-a9bb-0cdfe02e247a,208,27,Jack Massie
8,78c6e1c2-6f06-4cf7-8a82-54f62c0e9504,206,25,Rod McGeady
9,e2ffaf02-f17f-4be3-b3ee-651d39315af1,198,25,Mjd-⚒️⚒️⚒️


In [25]:
# Game-week player counts: list the game weeks used and player counts per week
import pandas as pd
from pathlib import Path
from IPython.display import display

if 'merged_all' not in globals():
    raise NameError("merged_all not found — run cell 1 first to load and merge game_week_scores, game_weeks, and profiles")

df = merged_all.copy()
base = Path('../CSV/06.04.2026')

# Ensure we have week_number; if not, try to load game_weeks and merge
if 'week_number' not in df.columns:
    gw_path = base / 'game_weeks_rows.csv'
    if gw_path.exists():
        gws = load_csv(gw_path)
        if 'game_week_id' in df.columns:
            df = df.merge(gws[['id','week_number','season_id']], left_on='game_week_id', right_on='id', how='left')
        elif 'game_week' in df.columns:
            df = df.merge(gws[['id','week_number','season_id']], left_on='game_week', right_on='id', how='left')

# Group by week_number and count distinct players
grouped = df.groupby(['week_number']).agg(players_in_week=('player_id', lambda x: x.nunique()), rows=('player_id','size')).reset_index()

# Attempt numeric sort of week_number when possible
try:
    grouped['week_number_num'] = pd.to_numeric(grouped['week_number'], errors='coerce')
    grouped = grouped.sort_values(['week_number_num'], na_position='last').drop(columns=['week_number_num']).reset_index(drop=True)
except Exception:
    grouped = grouped.sort_values('week_number', na_position='last').reset_index(drop=True)

print('Game weeks used and unique player counts:')
display(grouped)


Game weeks used and unique player counts:


,week_number,players_in_week,rows
0,1,35,35
1,2,34,34
2,3,34,34
3,4,34,34
4,5,34,34
5,6,34,34
6,7,34,34
7,8,34,34
8,9,34,34
9,10,34,34
